# Module 9: Deploy Your Agent on the Inference You Own

In Module 8 you benchmarked your server and wrote down an operating point you could defend. You tuned the inference layer until it was yours. Now you put an agent on top of it, and that is where the whole workshop pays off. You deploy a small Akamai Cloud solutions architect agent into your namespace as a plain Kubernetes Deployment, point it at the [vLLM](https://docs.vllm.ai) you have been tuning, talk to it, and then watch its traffic show up in the same metrics you read all day. The model answering every turn is the server you own, on your GPU, not a rented API.

## Learning objectives
- Deploy an agent as a plain Kubernetes Deployment and Service in your own namespace
- Point the agent at your in-namespace vLLM by its Service name, with no cross-namespace setup
- Reach the agent with kubectl port-forward and send it questions
- See the agent answer in scope and decline out of scope honestly
- Watch the agent's own load appear in your vLLM metrics, closing the loop on the workshop
- Know where this goes in production: scaling replicas and scaling nodes

## Prerequisites
- Finished the inference labs, especially `05_optimize_the_server`, with a working vLLM Service named `vllm` in your namespace
- A live cluster with `kubectl` and your namespace-scoped kubeconfig
- About 12 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [OpenAI chat API](https://platform.openai.com/docs/api-reference/chat) &middot; [Kubernetes Deployments](https://kubernetes.io/docs/concepts/workloads/controllers/deployment/) &middot; [kubectl port-forward](https://kubernetes.io/docs/reference/generated/kubectl/kubectl-commands#port-forward) &middot; [Akamai Cloud GPUs](https://www.linode.com/products/gpu/)

## Agent on owned inference design basics

An agent is just a client of a model. The only question this module answers is where that model lives. Here it lives in your namespace, on the GPU you tuned, and the agent is an ordinary container next to it.

- **A plain Deployment.** The agent is a small HTTP service (`agent/agent.py`): it wraps each request with a solutions-architect system prompt and calls your vLLM with the OpenAI client. No framework, no controller, no custom resources. It runs with a stock `python:3.12-slim` image and installs the one dependency it needs at start.
- **Same namespace, short name.** The agent and your vLLM share a namespace, so the agent reaches the model at the bare Service name `http://vllm:8000/v1`. There is no cross-namespace FQDN and nothing cluster-scoped to install, so your scoped kubeconfig can deploy the whole thing.
- **The persona.** The system prompt makes it an Akamai Cloud solutions architect: tactical, in scope on Akamai compute, LKE, storage, networking, GPUs, and inference, and honest about what it does not cover. It is the same persona the full Solutions Architect Agent workshop builds, kept to chat only so this capstone stays about inference.

![A small agent Deployment in your namespace calls your in-namespace vLLM Service on your GPU, and you reach the agent through kubectl port-forward](images/09_agents_on_k8s_architecture.png)

## 1. Setup

The notebook needs the OpenAI client and `requests` to talk to the agent and read metrics. Install them, then resolve your settings from the environment, the same `common/config.py` every module uses.

In [ ]:
%pip install -q "openai>=1.40" "requests>=2.31"

## 2. Your settings and the agent code

Resolve your namespace, vLLM URL, and model, then print the agent so you can read what you are about to deploy. It is short on purpose: a system prompt and one call to your vLLM.

In [ ]:
import os, sys, json, time, threading, subprocess, atexit
from concurrent.futures import ThreadPoolExecutor
sys.path.insert(0, os.path.abspath(".."))

from common.config import print_settings
from common import metrics

settings = print_settings()
NS = settings.namespace

print("\nThe agent you will deploy (agent/agent.py):")
print("-" * 60)
print(open("agent/agent.py").read())

**What you should see:** your settings, then the agent source. Note it reads `VLLM_BASE_URL` and `MODEL_NAME` from its environment and calls your vLLM with the OpenAI client. The persona in the system prompt is the whole personality; everything else is plumbing.

## 3. Deploy the agent

Two objects go into your namespace: a ConfigMap holding the agent code, and the Deployment plus Service that runs it (`manifests/agent.yaml`). The Deployment mounts the code, installs the OpenAI client at start, and serves on port 8080. All of it is namespaced, so your scoped kubeconfig is enough; there is nothing cluster-wide to install.

In [ ]:
# Requires your namespace-scoped kubeconfig.
# 1) Put the agent code in a ConfigMap the Deployment mounts at /app (idempotent).
cm = subprocess.run(
    ["kubectl", "create", "configmap", "agent-code",
     "--from-file=agent.py=agent/agent.py", "-n", NS,
     "--dry-run=client", "-o", "yaml"],
    check=True, capture_output=True, text=True).stdout
subprocess.run(["kubectl", "apply", "-f", "-", "-n", NS], input=cm, text=True, check=True)

# 2) Deploy the agent and its Service, then wait for it to be Ready.
subprocess.run(["kubectl", "apply", "-f", "manifests/agent.yaml", "-n", NS], check=True)
subprocess.run(["kubectl", "rollout", "status", "deploy/sa-agent", "-n", NS, "--timeout=180s"], check=True)
subprocess.run(["kubectl", "get", "pods", "-l", "app=agent", "-n", NS])

**What you should see:** the ConfigMap and Deployment applied, the rollout reaching Ready, and one `sa-agent` pod `Running`. The first start takes a few seconds to install the OpenAI client. If the pod is not Ready, `kubectl logs deploy/sa-agent -n $NS` shows why; a connection error to vLLM means your `vllm` Service is not up.

## 4. Talk to it

The agent has no public address, so port-forward its Service to your notebook and POST questions. Ask one thing in scope and one thing out of scope, and watch it route honestly instead of guessing. Every answer is generated by your vLLM.

In [ ]:
# Requires the agent Running. Port-forward the agent Service, then POST questions.
import requests

pf = subprocess.Popen(
    ["kubectl", "port-forward", "svc/sa-agent", "8080:8080", "-n", NS],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
atexit.register(pf.terminate)
time.sleep(4)  # let the port-forward establish

def ask(message):
    r = requests.post("http://localhost:8080", json={"message": message}, timeout=60)
    data = r.json()
    return data.get("answer") or data.get("error")

print("=== in scope ===")
print(ask("How do I create an LKE cluster with a GPU node pool?"))
print("\n=== out of scope ===")
print(ask("How do I write an EdgeWorker for the CDN?"))

**What you should see:** a tactical answer about LKE and GPU node pools for the first question, and a polite redirect to the edge compute team for the second. Both answers came from your vLLM, on your GPU. The port-forward stays open for the next cells.

## 5. Close the loop: see the agent's load in your metrics

This is the payoff of the whole workshop. Fire several agent questions at once and sample your vLLM metrics while they run. The agent's traffic is your vLLM's traffic: the same running batch and KV cache you have read and tuned since Module 2. The metrics you learned to operate are now reporting on a real agent.

In [ ]:
# Requires the agent up and the port-forward from section 4 still active.
peak = {"running": 0, "kv": 0.0}
stop = threading.Event()

def sample():
    while not stop.is_set():
        try:
            s = metrics.snapshot(settings.metrics_url)
            peak["running"] = max(peak["running"], int(s["vllm:num_requests_running"]))
            peak["kv"] = max(peak["kv"], s["vllm:gpu_cache_usage_perc"])
        except Exception:
            pass
        time.sleep(0.25)

questions = [
    "What is LKE?", "How do NodeBalancers work?", "When should I use Object Storage?",
    "What GPU plans does Akamai offer?", "How do I scope a Linode API token?",
    "What is a VPC on Akamai Cloud?",
]
t = threading.Thread(target=sample, daemon=True); t.start()
with ThreadPoolExecutor(max_workers=len(questions)) as pool:
    list(pool.map(ask, questions))
stop.set(); t.join(timeout=2)

print(f"peak vLLM requests running during the agent's load: {peak['running']}")
print(f"peak KV cache usage: {peak['kv']*100:.1f}%")
print("That load came from your agent, served by the vLLM you tuned.")

**What you should see:** a peak of several concurrent requests running on your vLLM and a bump in KV cache usage, driven entirely by the agent. The same continuous batching you watched form in Module 3 is now batching your agent's traffic.

## 6. In production: scaling and scale-to-zero

You now run two things you control: a vLLM serving the model and an agent calling it. In production you would not run them flat out around the clock. Two separate loops handle that, and it helps to keep them straight:

- **Scaling replicas.** Tools like KServe or Knative scale your inference and agent pods up under load and back down when traffic falls, including all the way to zero when nothing is calling. That trades a cold start for paying nothing while idle, which is the right deal for bursty or spiky traffic.
- **Scaling nodes.** A node autoscaler such as Karpenter or the cluster autoscaler adds and removes the GPU machines underneath, so when your pods scale to zero the expensive card can go away too.

This workshop deliberately did the opposite: one dedicated card, driven to saturation, tuned by hand, so you could see the limits. That is the right mode when you want predictable latency on steady traffic. Scale-to-zero is the other half of the story, for when the traffic is not steady. Same metrics, same vLLM, a different operating posture.

## Things to know

- **The agent is just a client.** It holds a system prompt and calls your vLLM with the OpenAI client. The lesson is where the model runs, not the agent code. Swap in any framework you like; the model stays yours.
- **Same namespace, short name.** Because the agent and vLLM share your namespace, the agent uses the short Service name `http://vllm:8000/v1`, with no cross-namespace FQDN and nothing cluster-scoped to install.
- **The agent is a new client through the firewall.** On a cluster that enforces a default-deny NetworkPolicy, your namespace needs a rule allowing `app: agent` to reach vLLM on 8000. The rule that lets your workspace reach vLLM does not cover the agent, because the agent is a different pod. On the hosted platform this belongs in the own-inference preset; if the agent cannot connect, that rule is what is missing.
- **Nothing here needs cluster admin.** A Deployment, a Service, and a ConfigMap all live in your namespace, which your scoped kubeconfig manages. You never touched another namespace or a custom resource.
- **The full agent is the next step.** This is the solutions-architect persona on chat only. The complete agent, with account-reading tools and memory, is the [Solutions Architect Agent workshop](https://github.com/akamai-developers/akamai-workshop-solution-architect-agent). Point its `VLLM_BASE_URL` at the vLLM you tuned here and it runs on inference you own.

> NOTE: vLLM enforces no API key by default, so the agent sends `not-needed`. If you put auth in front of your vLLM, set `VLLM_API_KEY` on the Deployment and the same agent keeps working.

## Try it yourself

**Ask it what powers it.** Send the agent "What model and endpoint are you running on?" and confirm it names your model and your vLLM. That is the agent reporting, in its own words, that it runs on inference you own.

**Tighten the scope.** Edit the system prompt in `agent/agent.py`, re-create the ConfigMap, and `kubectl rollout restart deploy/sa-agent`. Re-ask an out-of-scope question and watch the routing change.

**Give it a tool.** Add one function to `agent.py`, for example a call that lists your pods, and let the model decide when to use it. Now it does not just talk about your cluster, it reads it, still on the model you own.

In [ ]:
# Requires the agent up and the port-forward from section 4 active.
print(ask("What model and endpoint are you running on?"))

## Summary

- An agent is a client of a model. The whole point of this module is that its model is the vLLM you own, on your GPU, not a rented API.
- You deployed the agent as a plain Deployment and Service in your own namespace, with no CRDs, no controller, and no cluster admin.
- The agent reaches your vLLM by its short Service name, because they share a namespace, and you reached the agent with kubectl port-forward.
- You drove the agent under load and watched its traffic land in the same vLLM metrics you tuned all workshop. The agent's load is your load.
- In production you would scale replicas with KServe or Knative and nodes with an autoscaler; here you tuned one card on purpose, to see the limits.

## Done

You started by renting inference and ended with an agent answering on a GPU you tuned yourself. You own the whole stack now: the model, the server, the metrics, the tuning, and the agent on top of it.